# 👗 VTON - Beden Duyarlı Sanal Giydirme (Google Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zekiyeilman/VTON/blob/main/colab_run.ipynb)

> **Önce:** Çalışma zamanı → Çalışma zamanı türünü değiştir → **T4 GPU**

Hücreleri sırayla çalıştırın. Pip kurulumdan sonra *"Restart session"* isterse yeniden başlatıp **1. hücreden** devam edin.

## 1. Depoyu klonla / güncelle ve paketi kur

Repo **private** ise: sol menüdeki 🔑 **Secrets** bölümüne `GITHUB_TOKEN` adıyla bir GitHub token'ı ekleyin
(github.com → Settings → Developer settings → Fine-grained tokens → sadece bu repo, *Contents: Read-only*).
Repo public ise bu adım gerekmez.

In [ ]:
import os, sys

REPO = "/content/VTON"
REPO_PATH = "github.com/zekiyeilman/VTON.git"

try:
    from google.colab import userdata
    _gh = userdata.get("GITHUB_TOKEN")
except Exception:
    _gh = None
_url = f"https://{_gh}@{REPO_PATH}" if _gh else f"https://{REPO_PATH}"

if not os.path.exists(REPO):
    !git clone -q {_url} {REPO}
    # Token'ı kalıcı remote ayarında bırakma
    !git -C {REPO} remote set-url origin https://{REPO_PATH}
else:
    !git -C {REPO} pull -q {_url} main

%cd {REPO}
!pip install -q -e .

# Editable kurulum, çalışan kernel'de hemen görünmeyebilir; yolu elle ekliyoruz
if REPO not in sys.path:
    sys.path.insert(0, REPO)

## 2. Hugging Face token (opsiyonel)

HD render (IDM-VTON) için. Sol menüdeki 🔑 **Secrets** bölümüne `HF_TOKEN` adıyla ekleyin. Token koda **yazılmaz**.

In [ ]:
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN yüklendi.")
except Exception as e:
    print("HF_TOKEN bulunamadı (opsiyonel):", e)

## 3. GPU kontrolü

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 4. (Opsiyonel) Linkten veri çekmek için Chrome kurulumu

Scraper Colab'de **headless Chrome** ile çalışır. Sadece "Linkten Veri Çek" özelliğini kullanacaksanız gerekir.

In [ ]:
!wget -q -O /tmp/chrome.deb https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get install -y -qq /tmp/chrome.deb > /dev/null
!google-chrome --version

## 5. Modülleri kütüphane gibi kullanma

In [ ]:
import vton
from vton import Ansur3DTo2DConverter
from vton.paths import PROJECT_ROOT, goruntu

print("vton", vton.__version__, "->", PROJECT_ROOT)

ansur = Ansur3DTo2DConverter()
print("95 cm göğüs çevresi -> 2D ön genişlik:", round(ansur.circumference_to_width(95, "chest"), 1), "cm")

# Diğer modüller:
# from vton import VTONEngine, SegFormerNode, BodyMeshNode, FitHeatmapNode
# from vton.scraper import vton_otonom_boru_hatti

## 6. Gradio arayüzünü başlat (public link)

In [ ]:
import app
app.demo.launch(share=True, debug=True)